# OAGT_POD — Orbital Ambiguity Ground Track / Physical Observables Derivation

## 0. Setup
   - Imports, baseline TLE, ground station (Cal Poly), propagation utilities

## 1. Framework
   - Ranking, bounded vs unbounded sets, normalization approach
   - (Markdown — pulls from the spec we just wrote)

## 2. Per-element analysis
### 2.1 Δa  (semi-major axis)
### 2.2 Δi  (inclination)
### 2.3 ΔΩ  (RAAN)
### 2.4 ΔM  (mean anomaly)
### 2.5 ΔB* (drag)
### 2.6 Δe  (eccentricity)  [SSO: weak, brief check]
### 2.7 Δω  (argument of perigee) [SSO: weak, brief check]

## 3. Cross-element comparison
   - Visual ranking validation: side-by-side ground tracks at fixed Δ%
   - Confirm/revise qualitative ranking

## 4. Threshold derivation
   - Per element, what perturbation magnitude is operator-detectable?
   - This is what feeds back into pass_ambiguity.md

## 5. Open questions / next steps

In [23]:
# =============================================================================
# Section 0 — Setup
# OAGT_POD: Orbital Ambiguity Ground Track / Physical Observables Derivation
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from astropy.time import Time
from astropy import units as u
from astropy.coordinates import (TEME, ITRS, EarthLocation,
                                  CartesianRepresentation)
from sgp4.api import Satrec, WGS72
from datetime import datetime, timezone

# ---- 0.1 Constants ----------------------------------------------------------
MU_EARTH = 398600.4418    # km^3/s^2
R_EARTH  = 6378.137       # km
DEG2RAD  = np.pi / 180.0
RAD2DEG  = 180.0 / np.pi

# ---- 0.2 Ground station (Cal Poly SLO) --------------------------------------
GS_NAME    = "Cal Poly SLO"
GS_LAT     = 35.30         # deg N
GS_LON     = -120.66       # deg E
GS_ALT     = 0.097         # km
MIN_ELEV   = 10.0          # deg
GS_LAT_RAD = np.radians(GS_LAT)
GS_LON_RAD = np.radians(GS_LON)

print("Imports and constants loaded.")
print(f"Ground station: {GS_NAME} ({GS_LAT}°N, {GS_LON}°E)")

Imports and constants loaded.
Ground station: Cal Poly SLO (35.3°N, -120.66°E)


In [24]:
# ---- 0.3 Load historical TLE cache ------------------------------------------
cache_dir = os.path.expanduser('~/mission_cache')

with open(os.path.join(cache_dir, '2023-174_history.json')) as f:
    cache = json.load(f)

hist_df = pd.DataFrame(cache['tles'])
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)

print(f"hist_df for T-9: {len(hist_df)} records, {hist_df['norad'].nunique()} satellites")

hist_df for T-9: 9973 records, 92 satellites


In [25]:
# ---- 0.4 Extract orbital elements from hist_df ------------------------------
launch_date = '2023-11-11'
launch_ts   = pd.Timestamp(launch_date, tz='UTC')

def parse_bstar(s):
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            return float('0.' + s[:i].lstrip('+-')) * (10 ** int(s[i:]))
    return float(s)

def extract_elements(line1, line2):
    inc   = float(line2[8:16])
    raan  = float(line2[17:25])
    ecc   = float('0.' + line2[26:33].strip())
    argp  = float(line2[34:42])
    M     = float(line2[43:51])
    n     = float(line2[52:63]) * 2 * np.pi / 86400.0
    a_km  = (MU_EARTH / n**2) ** (1/3)
    bstar = parse_bstar(line1[53:61])
    return dict(a_km=a_km, inc=inc, raan=raan,
                ecc=ecc, argp=argp, M=M, bstar=bstar)

rows = []
for _, row in hist_df.iterrows():
    try:
        el = extract_elements(row['line1'], row['line2'])
    except Exception:
        continue
    el['norad']             = int(row['norad'])
    el['name']              = row['name']
    el['epoch']             = row['epoch']
    el['days_since_launch'] = (row['epoch'] - launch_ts).total_seconds() / 86400
    rows.append(el)

elements_df = pd.DataFrame(rows)
print(f"elements_df: {len(elements_df)} rows, {elements_df['norad'].nunique()} satellites")

elements_df: 9973 rows, 92 satellites


In [26]:
# ---- 0.5 Select baseline satellite (nearest cluster median in a, i) ---------
latest_per_sat = (elements_df.sort_values('days_since_launch')
                              .groupby('norad', as_index=False)
                              .last())

median_a = latest_per_sat['a_km'].median()
median_i = latest_per_sat['inc'].median()

norm_dist = (
    ((latest_per_sat['a_km'] - median_a) / latest_per_sat['a_km'].std())**2 +
    ((latest_per_sat['inc']  - median_i) / latest_per_sat['inc'].std())**2
) ** 0.5

baseline_idx   = norm_dist.idxmin()
baseline_row   = latest_per_sat.loc[baseline_idx]
baseline_norad = int(baseline_row['norad'])

print(f"Baseline satellite: {baseline_row['name']}  (NORAD {baseline_norad})")
print(f"  a    = {baseline_row['a_km']:.3f} km  (cluster median {median_a:.3f})")
print(f"  i    = {baseline_row['inc']:.4f} deg  (cluster median {median_i:.4f})")
print(f"  RAAN = {baseline_row['raan']:.4f} deg")
print(f"  e    = {baseline_row['ecc']:.6f}")
print(f"  ω    = {baseline_row['argp']:.4f} deg")
print(f"  M    = {baseline_row['M']:.4f} deg")
print(f"  B*   = {baseline_row['bstar']:.4e}")

Baseline satellite: ICEYE-X31  (NORAD 58288)
  a    = 6898.143 km  (cluster median 6898.064)
  i    = 97.4791 deg  (cluster median 97.4768)
  RAAN = 86.8576 deg
  e    = 0.001108
  ω    = 68.6482 deg
  M    = 291.5931 deg
  B*   = 6.3251e-04


In [27]:
# ---- 0.6 Load actual TLE lines for baseline ---------------------------------
baseline_tle = (hist_df[hist_df['norad'].astype(int) == baseline_norad]
                .sort_values('epoch')
                .iloc[-1])

BASELINE_NAME  = baseline_tle['name']
BASELINE_LINE1 = baseline_tle['line1']
BASELINE_LINE2 = baseline_tle['line2']

baseline_sat = Satrec.twoline2rv(BASELINE_LINE1, BASELINE_LINE2)

print(f"Baseline TLE loaded: {BASELINE_NAME}")
print(f"  Epoch: {baseline_tle['epoch']}")
print(f"  Line1: {BASELINE_LINE1}")
print(f"  Line2: {BASELINE_LINE2}")

Baseline TLE loaded: ICEYE-X31
  Epoch: 2024-01-09 21:12:55.893600+00:00
  Line1: 1 58288U 23174AJ  24009.88398025  .00011879  00000-0  63251-3 0  9992
  Line2: 2 58288  97.4791  86.8576 0011076  68.6482 291.5931 15.15320315  9554


# Perturbation Model for single orbital element on BASELINE TLE: ICEYE-X31
- We are making a perturbation model for each physical orbital element. Lets make it percentage of the natural domain of each orbital element. We will have a ground track display and on this ground track display, we will have 3-5 different traces? We are going to go per-element, keeping all other orbital elements constant. We are color coding each orbital trace, so one of them is the baseline trace (no perturbation), then 3-5 more with incrementally larger perturbations on that specific element. 

| Element | Domain | 1% | 5% | 10% | 25% |
|---|---:|---:|---:|---:|---:|
| $i$ | $180^\circ$ | $1.8^\circ$ | $9^\circ$ | $18^\circ$ | $45^\circ$ |
| $\Omega$ | $360^\circ$ | $3.6^\circ$ | $18^\circ$ | $36^\circ$ | $90^\circ$ |
| $M$ | $360^\circ$ | $3.6^\circ$ | $18^\circ$ | $36^\circ$ | $90^\circ$ |
| $\omega$ | $360^\circ$ | $3.6^\circ$ | $18^\circ$ | $36^\circ$ | $90^\circ$ |
| $e$ | $[0,1]$ | $0.01$ | $0.05$ | $0.10$ | $0.25$ |
| $a$ | unbounded — use km | $1\text{ km}$ | $5\text{ km}$ | $10\text{ km}$ | $50\text{ km}$ |
| $B^*$ | unbounded — use $\times$ factor | $\times 2$ | $\times 5$ | $\times 10$ | $\times 50$ |

- For propagation duration, we will go based on element (for ex. RAAN may have slower accumulative effects for drift). For starting epoch we will use Jan 9 2024, it doesn't really matter I feel like because the whole point is to see perturbative effects which is sort of TLE-independent. I want the algorithm for make_variant to be copy-pastable so like first element i we add perturbative effects by percentage of the natural domain, not the actual values.


In [28]:
# ---- 0.6 make_variant — percentage-based perturbation helper ----------------
# Usage: make_variant(baseline_sat, 'i', 5.0)
# Returns a new Satrec with inclination perturbed by 5% of 180°, all else fixed.

DOMAINS = {
    'i':    180.0,   # degrees
    'raan': 360.0,   # degrees
    'M':    360.0,   # degrees
    'argp': 360.0,   # degrees
    'e':    1.0,     # dimensionless [0,1]
    # 'a' and 'bstar' handled separately (unbounded)
}

def make_variant(baseline_satrec, element, pct):
    """
    Perturb one orbital element by pct% of its natural domain.
    element: one of 'i', 'raan', 'M', 'argp', 'e'
    pct:     percentage of natural domain (e.g. 5.0 = 5%)
    Returns: new Satrec, delta applied in natural units
    """
    assert element in DOMAINS, f"Use make_variant_km() for 'a', make_variant_bstar() for 'bstar'"
    delta = (pct / 100.0) * DOMAINS[element]

    s = baseline_satrec
    # Pull current elements
    n_rad_s      = s.no_kozai / 60.0           # rad/s
    a_km         = (MU_EARTH / n_rad_s**2) ** (1/3)
    i_rad        = s.inclo
    raan_rad     = s.nodeo
    ecc          = s.ecco
    argp_rad     = s.argpo
    M_rad        = s.mo
    bstar        = s.bstar

    # Apply perturbation
    if element == 'i':    i_rad    += np.radians(delta)
    if element == 'raan': raan_rad += np.radians(delta)
    if element == 'M':    M_rad    += np.radians(delta)
    if element == 'argp': argp_rad += np.radians(delta)
    if element == 'e':    ecc      += delta

    # Recompute mean motion
    n_rad_s_new    = np.sqrt(MU_EARTH / a_km**3)
    no_kozai_new   = n_rad_s_new * 60.0         # rad/min

    new = Satrec()
    new.sgp4init(
        WGS72, 'i',
        s.satnum,
        s.jdsatepoch - 2433281.5 + s.jdsatepochF,
        bstar,
        s.ndot, s.nddot,
        ecc, argp_rad, i_rad, M_rad, no_kozai_new, raan_rad,
    )
    return new, delta

def make_variant_km(baseline_satrec, delta_a_km):
    """Perturb semi-major axis by delta_a_km (signed, km)."""
    s = baseline_satrec
    n_rad_s  = s.no_kozai / 60.0
    a_km     = (MU_EARTH / n_rad_s**2) ** (1/3) + delta_a_km
    n_new    = np.sqrt(MU_EARTH / a_km**3) * 60.0

    new = Satrec()
    new.sgp4init(
        WGS72, 'i', s.satnum,
        s.jdsatepoch - 2433281.5 + s.jdsatepochF,
        s.bstar, s.ndot, s.nddot,
        s.ecco, s.argpo, s.inclo, s.mo, n_new, s.nodeo,
    )
    return new, delta_a_km

def make_variant_bstar(baseline_satrec, factor):
    """Perturb B* by multiplying by factor (e.g. factor=5 → B* × 5)."""
    s = baseline_satrec
    n_rad_s = s.no_kozai / 60.0

    new = Satrec()
    new.sgp4init(
        WGS72, 'i', s.satnum,
        s.jdsatepoch - 2433281.5 + s.jdsatepochF,
        s.bstar * factor, s.ndot, s.nddot,
        s.ecco, s.argpo, s.inclo, s.mo, s.no_kozai, s.nodeo,
    )
    return new, s.bstar * factor

print("make_variant / make_variant_km / make_variant_bstar loaded.")
print("Usage: make_variant(baseline_sat, 'i', 5.0)  → perturb i by 5% of 180°")

make_variant / make_variant_km / make_variant_bstar loaded.
Usage: make_variant(baseline_sat, 'i', 5.0)  → perturb i by 5% of 180°


In [29]:
# ---- 0.7 Propagation + ground track + elevation helpers --------------------

# Propagation start epoch — ICEYE-X31 TLE epoch (Jan 9 2024)
from astropy.time import Time
PROP_START = Time("2024-01-09T21:12:55", format='isot', scale='utc')

def propagate(satrec, n_hours, n_steps):
    """
    Propagate satrec over n_hours from PROP_START.
    Returns: lats, lons, alts, elevations (all shape n_steps)
    """
    times  = PROP_START + np.linspace(0, n_hours, n_steps) * u.hour
    jd1    = np.array([t.jd1 for t in times])
    jd2    = np.array([t.jd2 for t in times])
    err, r, v = satrec.sgp4_array(jd1, jd2)

    lats, lons, alts = [], [], []
    for j in range(n_steps):
        if err[j] != 0:
            lats.append(np.nan); lons.append(np.nan); alts.append(np.nan)
            continue
        cart = CartesianRepresentation(r[j] * u.km)
        teme = TEME(cart, obstime=times[j])
        itrs = teme.transform_to(ITRS(obstime=times[j]))
        loc  = EarthLocation(*itrs.cartesian.xyz)
        lats.append(loc.lat.deg)
        lons.append(loc.lon.deg)
        alts.append(loc.height.to(u.km).value)

    lats = np.array(lats)
    lons = np.array(lons)
    alts = np.array(alts)

    # Elevation from Cal Poly
    lat_r  = np.radians(lats)
    lon_r  = np.radians(lons)
    cos_sep = (np.sin(GS_LAT_RAD)*np.sin(lat_r) +
               np.cos(GS_LAT_RAD)*np.cos(lat_r)*np.cos(lon_r - GS_LON_RAD))
    sep    = np.arccos(np.clip(cos_sep, -1, 1))
    r_sat  = R_EARTH + alts
    elevs  = np.degrees(np.arctan2(np.cos(sep) - R_EARTH/r_sat, np.sin(sep)))

    time_hrs = np.linspace(0, n_hours, n_steps)
    return lats, lons, alts, elevs, time_hrs

print("propagate() loaded.")
print("Usage: lats, lons, alts, elevs, t_hrs = propagate(satrec, n_hours=24, n_steps=2000)")

propagate() loaded.
Usage: lats, lons, alts, elevs, t_hrs = propagate(satrec, n_hours=24, n_steps=2000)


- I need the graph to be movable, and so I can zoom in. Also, we should put a visibility circle that accounts for the elevation arc of the satellite over the horizon with a minimum of 10 degrees. This way I can see when the satellite is in view of the ground station, and verify if perturbations stay within this window. 

In [30]:
# ───────────────────────────────────────────────────────────────────
# OAGT_POD plot_element_perturbations — v2-style
# Propagates baseline + N perturbations, splits each into orbits,
# plots ground tracks (geo) and elevation arcs (xy), with dropdown.
# ───────────────────────────────────────────────────────────────────

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np


# ── Helpers ────────────────────────────────────────────────────────
def break_antimeridian(lons):
    """Null out longitudes where the jump exceeds 180° (Plotly artifact)."""
    lons_plot = np.asarray(lons, dtype=float).copy()
    for i in range(1, len(lons_plot)):
        if abs(lons_plot[i] - lons_plot[i-1]) > 180:
            lons_plot[i-1] = np.nan
    return lons_plot


def compute_visibility_circle(gs_lat_deg, gs_lon_deg, alt_km,
                               min_elev_deg=10.0, n_pts=361):
    """Great-circle ring at which elevation = min_elev for satellites at alt_km."""
    R_e = R_EARTH
    elev_rad = np.radians(min_elev_deg)
    rho = np.arccos(R_e / (R_e + alt_km)) - elev_rad

    lat0 = np.radians(gs_lat_deg)
    lon0_deg = gs_lon_deg
    az = np.linspace(0, 2*np.pi, n_pts)

    circ_lats = np.degrees(np.arcsin(
        np.sin(lat0)*np.cos(rho) + np.cos(lat0)*np.sin(rho)*np.cos(az)
    ))
    circ_lons = lon0_deg + np.degrees(np.arctan2(
        np.sin(az)*np.sin(rho)*np.cos(lat0),
        np.cos(rho) - np.sin(lat0)*np.sin(np.radians(circ_lats))
    ))
    return circ_lats, circ_lons


def split_into_orbits(arrays, period_min, n_hours, n_steps):
    """
    arrays: dict of np.ndarray, all length n_steps
    Returns list of dicts (one per orbit) with the same keys, sliced.
    """
    period_s   = period_min * 60.0
    duration_s = n_hours * 3600.0
    period_steps = max(1, int(round(n_steps / (duration_s / period_s))))
    n_orbits = int(np.ceil(n_steps / period_steps))

    orbits = []
    for k in range(n_orbits):
        s = k * period_steps
        e = min((k + 1) * period_steps, n_steps)
        if e - s < 2:                              # skip stub
            continue
        orbit = {key: arr[s:e] for key, arr in arrays.items()}
        orbit['number'] = k + 1
        orbits.append(orbit)
    return orbits


# ── Variant colors (baseline + 4 perturbations) ────────────────────
# White baseline so it visually anchors as the reference; perturbations
# walk the spectrum from cool (small) to hot (large).
VARIANT_COLORS = ['#FFFFFF', '#00FFFF', '#00FF00', '#FFD700', '#FF4500']


# ── Main plotting function ─────────────────────────────────────────
def plot_element_perturbations(element, pcts, n_hours=24, n_steps=2000):
    # Build variants
    satrecs = [baseline_sat] + [make_variant(baseline_sat, element, p)[0]
                                 for p in pcts]
    labels  = ['Baseline'] + [f'+{p}% of domain' for p in pcts]
    colors  = VARIANT_COLORS[:len(satrecs)]
    domain  = DOMAINS[element]

    # Baseline orbital period & altitude (for orbit-splitting & visibility)
    T_min      = 2 * np.pi / baseline_sat.no_kozai          # rad/min → minutes
    n_rad_s    = baseline_sat.no_kozai / 60.0
    base_alt   = (MU_EARTH / n_rad_s**2) ** (1/3) - R_EARTH

    # Propagate every variant
    variant_orbits = []
    for sat in satrecs:
        lats, lons, alts, elevs, t_hrs = propagate(sat, n_hours, n_steps)
        orbits = split_into_orbits(
            {'lats': lats, 'lons': lons, 'alts': alts,
             'elevs': elevs, 't_hrs': t_hrs},
            period_min=T_min, n_hours=n_hours, n_steps=n_steps,
        )
        variant_orbits.append(orbits)

    n_orbits = max(len(v) for v in variant_orbits)
    print(f"Propagated {len(satrecs)} variants × ~{n_orbits} orbits each")

    # Visibility circle (uses baseline altitude)
    vis_lats, vis_lons = compute_visibility_circle(
        GS_LAT, GS_LON, base_alt, MIN_ELEV
    )

    # ── Build figure ───────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.65, 0.35],
        specs=[[{'type': 'geo'}], [{'type': 'xy'}]],
        subplot_titles=[
            f'Ground tracks — Δ{element} (domain = {domain}°), '
            f'baseline alt ≈ {base_alt:.0f} km',
            f'Elevation from {GS_NAME}  (dashed = {MIN_ELEV}° min)'
        ],
        vertical_spacing=0.10,
    )

    # Track which trace indices belong to which (variant, orbit)
    # so we can build the dropdown visibility arrays.
    trace_meta = []   # list of (variant_idx, orbit_idx) for ground-track traces
    elev_meta  = []   # list of (variant_idx, orbit_idx) for elevation traces

    # Ground-track traces (one per variant per orbit)
    for v_idx, (orbits, color, label) in enumerate(zip(variant_orbits, colors, labels)):
        for o_idx, orb in enumerate(orbits):
            fig.add_trace(go.Scattergeo(
                lon=break_antimeridian(orb['lons']),
                lat=orb['lats'],
                mode='lines',
                line=dict(color=color, width=2.0 if v_idx == 0 else 1.3),
                name=f'{label} — Orbit {orb["number"]}',
                legendgroup=label,
                showlegend=False,     # one legend entry per variant
                hovertemplate=f'{label}<br>Orbit {orb["number"]}<extra></extra>',
            ), row=1, col=1)
            trace_meta.append((v_idx, o_idx))

    # Elevation traces (one per variant per orbit)
    for v_idx, (orbits, color, label) in enumerate(zip(variant_orbits, colors, labels)):
        for o_idx, orb in enumerate(orbits):
            fig.add_trace(go.Scatter(
                x=orb['t_hrs'], y=orb['elevs'],
                mode='lines',
                line=dict(color=color, width=2.0 if v_idx == 0 else 1.3),
                name=f'{label} — Orbit {orb["number"]}',
                legendgroup=label,
                showlegend=False,
                hovertemplate=f'{label}<br>Orbit {orb["number"]}<br>'
                              't=%{x:.2f} hr<br>el=%{y:.1f}°<extra></extra>',
            ), row=2, col=1)
            elev_meta.append((v_idx, o_idx))

    # ── Proxy legend traces (one per variant, always visible) ──────
# These exist only to provide a clickable legend entry that toggles
# all orbits of a variant via legendgroup. lat=[None] keeps them
# from rendering anything on the map.
    for color, label in zip(colors, labels):
        fig.add_trace(go.Scattergeo(
            lat=[None], lon=[None],
            mode='lines',
            line=dict(color=color, width=3),
            name=label,
            legendgroup=label,
            showlegend=True,
            hoverinfo='skip',
        ), row=1, col=1)

    n_proxy = len(satrecs)

    # ── Static traces (always visible) ─────────────────────────────
    static_idx_start = len(fig.data)

    fig.add_trace(go.Scattergeo(
        lat=vis_lats, lon=vis_lons, mode='lines',
        line=dict(color='orange', width=1.5, dash='dot'),
        name=f'Visibility circle ({MIN_ELEV}°)',
    ), row=1, col=1)

    fig.add_trace(go.Scattergeo(
        lat=[GS_LAT], lon=[GS_LON],
        mode='markers+text',
        marker=dict(size=10, color='orange', symbol='triangle-up'),
        text=[GS_NAME], textposition='top right',
        textfont=dict(color='white', size=11),
        showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=[0, n_hours], y=[MIN_ELEV, MIN_ELEV],
        mode='lines',
        line=dict(color='gray', dash='dash', width=1),
        showlegend=False,
    ), row=2, col=1)

    n_static = len(fig.data) - static_idx_start

    # ── Dropdown: filter by orbit number ───────────────────────────
    n_dyn = len(trace_meta) + len(elev_meta)

    def visibility_for(orbit_filter):
        """orbit_filter: None for all, otherwise a 1-indexed orbit number."""
        vis = []
        # Ground-track traces
        for (v_idx, o_idx) in trace_meta:
            if orbit_filter is None:
                vis.append(True)
            else:
                vis.append(variant_orbits[v_idx][o_idx]['number'] == orbit_filter)
        # Elevation traces
        for (v_idx, o_idx) in elev_meta:
            if orbit_filter is None:
                vis.append(True)
            else:
                vis.append(variant_orbits[v_idx][o_idx]['number'] == orbit_filter)
        # Static
        # Proxies (always visible) + static
        vis += [True] * (n_proxy + n_static)
        return vis

    buttons = [dict(
        label='All Orbits',
        method='update',
        args=[{'visible': visibility_for(None)}],
    )]
    for k in range(1, n_orbits + 1):
        buttons.append(dict(
            label=f'Orbit {k}',
            method='update',
            args=[{'visible': visibility_for(k)}],
        ))

    # ── Layout ─────────────────────────────────────────────────────
    fig.update_geos(
        projection_type='natural earth',
        showland=True,     landcolor='darkgreen',
        showocean=True,    oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True,  countrycolor='gray',
        showframe=False,
        bgcolor='black',
        center=dict(lat=GS_LAT, lon=GS_LON),
    )
    fig.update_xaxes(title_text='Hours from epoch',
                     gridcolor='gray', row=2, col=1)
    fig.update_yaxes(title_text='Elevation (°)',
                     gridcolor='gray', range=[-10, 95], row=2, col=1)

    fig.update_layout(
        title=f'OAGT_POD — Δ{element} | Baseline: {BASELINE_NAME}',
        height=850,
        dragmode='zoom',
        paper_bgcolor='black',
        plot_bgcolor='black',
        font=dict(color='white'),
        legend=dict(bgcolor='black', x=1.02, y=1.0),
        margin=dict(r=180),
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.02, y=1.08,
            bgcolor='black',
            font=dict(color='white'),
        )],
    )
    fig.show()

# Inclination Peturbations

In [31]:
plot_element_perturbations('i', pcts=[1, 5, 10, 25], n_hours=24)

Propagated 5 variants × ~16 orbits each


In [33]:
plot_element_perturbations('i', pcts=[1, 2, 3, 4], n_hours=24)

Propagated 5 variants × ~16 orbits each


## Scaling physical thresholds with GS location

### Which GS properties matter, and for which elements?
- **Δi, ΔΩ**: thresholds scale with GS latitude. Ground track 
  separation from an inclination perturbation is ~zero at equatorial 
  node crossings and maximal at high latitude. A threshold derived 
  at 35°N (Cal Poly) is conservative relative to equatorial stations 
  and optimistic relative to polar ones.
- **ΔM, Δa, ΔB***: thresholds are GS-location independent. These 
  elements affect timing or altitude, not whether the track reaches 
  the GS at all.

### Architectural implication
Layer 1 remains operator-independent in its *computation* — it scores 
orbital element differences without any GS input. The *thresholds* that 
convert those scores to "ambiguous / distinguishable" decisions are 
calibrated at a reference GS latitude (35°N). Layer 2 is then the 
correction term: it refines the ambiguity assessment for the specific 
team's GS location and pass geometry.

This gives Layer 2 a stronger motivation than "does the track reach 
the visibility window" — it is the operator-specific correction to a 
GS-agnostic Layer 1 triage score.


# Deriving analytical function for displacement of the ground track trace as a function of the latitude of ground-station if baseline TLE element is perturbed a certain %
## **Derivation** 
For a circular orbit at inclination $i$, the ground track satisfies the spherical triangle relation: $$ \text{sin } \theta = \text{sin } i \cdot \text{sin } u$$
where $\phi$ is latitude and $u$ is argument of latitude (angle from ascending nodes). Differentiating with respect to $i$ at fixed $u$: $$ \text{cos }\phi \cdot \frac{d\phi}{di} = \text{cos }i \cdot \text{sin }u$$ Substituting sin $u$ = sin $\phi$ / sin $i$: $$ \frac{d\phi}{di} = \frac{\text{tan }\phi}{\text{tan }i} $$ So the cross-track ground displacement in km for a peturbation $\Delta i$ is: $$\Delta d_\perp(\phi) = R_E \cdot \frac{|\text{tan }\phi|}{|\text{tan }i|} \cdot |\Delta i| $$ 

### What this tells us
- The scaling factor for the cross-track displacement for a perturbation $\Delta i$ is tan $\phi$ / tan $i$. 
#### Limit Tests:
- $\phi \rightarrow 0$(equator, near node crossing): displacement $\rightarrow 0$. Tracks converge at the nodes regardless of the inclination. This looks like a pinch point for all perturbed traces. 

- $\phi \rightarrow i$(maximum latitude, turning point): tan $\phi$ / tan $i \rightarrow 1$, so displacement $= R_e \Delta i$. This is the maximum sensitvity point. 

- $i \rightarrow 90^\circ$ (polar orbit): |tan $i$| $\rightarrow \inf$, so displacement $\rightarrow 0$ for all latitudes. Near-polar orbits are nearly insensitive to small inclination perturbation. 


## For May 4th 2026
- so going back to inclination. From a single orbital element perspective, we can assume a certain ambiguity based on the delta of inclination between two satellites. If the delta is below a certain physical threshold (we still have to find a good candiate physical threshold), we identify the pair as a having inclination ambiguity. The next step is cross-analyzing this with the intended ground station location, where we have a function with input (ground station location) and the output is a perpendicular distance of displacement. This gives us a better understanding of difference between the trajectories over the pass window. We need a measure to determine if it is still ambiguous over the pass window, and if it is then we can use doppler shift based on the satellite's angle of approach, or other things like mean anomaly. Another thing to consider is the ambiguity measure based on multiple ambiguous deltas. But this can be discussed after deriving all single element ambiguity tree properties. 

## ** Before Moving on** Read Claude's output after reading this summary, also read analytical function derivation, go back and review perturbation, also make sure to add notes and structure the original main files 
- Also cross-check this summary with over arching goal (discriminator, operator decisions, etc...)
- 

# RAAN Perturbations 

In [34]:
plot_element_perturbations('raan', pcts=[1, 5, 10, 25], n_hours=24)

Propagated 5 variants × ~16 orbits each


Deriving analytical function for $\Delta \Omega$ using RAAN perturbation as input: 

For $\Delta \Omega$ the cross-track displacement as latitude $\phi$ is approximately:
$\Delta d_\perp (\phi) = R_E \cdot |\Delta \Omega| \cdot \text{cos } \phi$

Note the key difference from the inclination formula: 

|Element| Scaling Factor | Maximum Sensitivity|
|-------|----------------|--------------------|
| $\Delta i$| $\frac{\text{tan } \phi}{\text{tan }i}$| High Latitude|
| $\Delta \Omega$ | cos $\phi$ | Equator |$

The two elements $\Delta i \text{ and } \Delta \Omega$ are **complementary discriminators**. A pair that is hard to separate by inclination is easier by RAAN, and vice versa. Inclination (North-South) changes the amplitude of the sinusoidal trace, and RAAN represents more of a phase shift (East-West). 

In [37]:
plot_element_perturbations('a', pcts=[1, 5, 10, 25], n_hours=24)

AssertionError: Use make_variant_km() for 'a', make_variant_bstar() for 'bstar'